<a href="https://colab.research.google.com/github/kamal-gavel/LSTM-Based-Stock-Return-Prediction-A-Normalization-Perspective-BatchNorm-vs-LayerNorm-/blob/main/LSTM_Based_Stock_Return_Prediction_A_Normalization_Perspective_BatchNorm_vs_LayerNorm_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================
# 📊 FINAL STOCK PREDICTION PIPELINE (NO SENTIMENT)
# ============================================

"""
THEORY:
--------
1. Stock prices are non-stationary → use RETURNS
   r_t = log(P_t / P_{t-1})

2. Normalization:
   - NoNorm: baseline
   - BatchNorm: not suitable for time series
   - LayerNorm: best for LSTM

3. Evaluation:
   - RMSE → error
   - R² → explanatory power
   - Directional Accuracy → trading relevance
   - PnL → economic significance
"""

# ==============================
# 🔧 INSTALL
# ==============================
!pip install yfinance ta -q

# ==============================
# 🔧 IMPORTS
# ==============================
import numpy as np
import pandas as pd
import yfinance as yf
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import ta

# ==============================
# 🔧 DEVICE
# ==============================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================
# 🔧 PARAMETERS
# ==============================
SEQ_LEN = 60
EPOCHS = 15
HORIZONS = [1, 3, 5]

STOCKS = {
    "RELIANCE": "RELIANCE.NS",
    "HDFCBANK": "HDFCBANK.NS",
    "INFOSYS": "INFY.NS"
}

# ==============================
# 🔶 MODEL
# ==============================
class LSTM_Model(nn.Module):
    """
    LSTM Model with normalization options
    """
    def __init__(self, input_size, norm=None):
        super().__init__()
        self.lstm = nn.LSTM(input_size, 64, batch_first=True)

        if norm == "BN":
            self.norm = nn.BatchNorm1d(64)
        elif norm == "LN":
            self.norm = nn.LayerNorm(64)
        else:
            self.norm = None

        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]

        if self.norm:
            out = self.norm(out)

        return self.fc(out)

# ==============================
# 🔶 TRAIN FUNCTION
# ==============================
def train_model(model, X, y):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()

    for _ in range(EPOCHS):
        model.train()
        optimizer.zero_grad()

        preds = model(X)
        loss = loss_fn(preds, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    return model

# ==============================
# 🔶 METRICS
# ==============================
def directional_accuracy(y_true, y_pred):
    return np.mean(np.sign(y_true) == np.sign(y_pred))

def trading_pnl(y_true, y_pred):
    signals = np.sign(y_pred)
    return np.sum(signals * y_true)

# ==============================
# 🔶 MAIN PIPELINE
# ==============================
for name, ticker in STOCKS.items():

    print(f"\n🚀 Processing {name}")

    # --------------------------
    # 🔹 LOAD DATA
    # --------------------------
    df = yf.download(ticker, start="2015-01-01", auto_adjust=True)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.droplevel(1)

    df = df[['Close']].dropna()

    # --------------------------
    # 🔥 RETURNS (CRITICAL)
    # --------------------------
    df['returns'] = np.log(df['Close'] / df['Close'].shift(1))

    # --------------------------
    # 🔹 TECHNICAL INDICATORS
    # --------------------------
    df['rsi'] = ta.momentum.RSIIndicator(df['Close']).rsi()
    df['macd'] = ta.trend.MACD(df['Close']).macd()
    df['bb'] = ta.volatility.BollingerBands(df['Close']).bollinger_mavg()

    df = df.dropna()

    print(f"{name} usable rows: {len(df)}")

    # --------------------------
    # 🔹 FEATURES
    # --------------------------
    FEATURES = ['returns', 'rsi', 'macd', 'bb']

    data = df[FEATURES].values

    # --------------------------
    # 🔹 SCALING
    # --------------------------
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)

    # ==========================
    # 🔶 MULTI-HORIZON LOOP
    # ==========================
    for H in HORIZONS:

        print(f"\n📊 Horizon t+{H}")

        X, y = [], []

        for i in range(SEQ_LEN, len(df) - H):
            X.append(data_scaled[i-SEQ_LEN:i])
            y.append(df['returns'].iloc[i + H])

        X = np.array(X)
        y = np.array(y)

        if len(X) == 0:
            print("⚠️ Skipping (no data)")
            continue

        # --------------------------
        # 🔹 TRAIN-TEST SPLIT
        # --------------------------
        split = int(0.8 * len(X))
        X_train, X_test = X[:split], X[split:]
        y_train, y_test = y[:split], y[split:]

        X_train = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
        y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(DEVICE)
        X_test = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
        y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1).to(DEVICE)

        # --------------------------
        # 🔹 MODELS
        # --------------------------
        models = {
            "NoNorm": LSTM_Model(len(FEATURES), None),
            "BatchNorm": LSTM_Model(len(FEATURES), "BN"),
            "LayerNorm": LSTM_Model(len(FEATURES), "LN")
        }

        # --------------------------
        # 🔹 TRAIN & EVALUATE
        # --------------------------
        for mname, model in models.items():

            model = train_model(model, X_train, y_train)

            model.eval()
            with torch.no_grad():
                preds = model(X_test).cpu().numpy().flatten()
                true = y_test.cpu().numpy().flatten()

            rmse = np.sqrt(mean_squared_error(true, preds))
            r2 = r2_score(true, preds)
            da = directional_accuracy(true, preds)
            pnl = trading_pnl(true, preds)

            print(f"{mname}: RMSE={rmse:.4f}, R2={r2:.4f}, DA={da:.4f}, PnL={pnl:.4f}")

  Preparing metadata (setup.py) ... done

🚀 Processing RELIANCE


[*********************100%***********************]  1 of 1 completed


RELIANCE usable rows: 2750

📊 Horizon t+1
NoNorm: RMSE=0.0155, R2=-0.3034, DA=0.5242, PnL=-0.0872
BatchNorm: RMSE=0.0632, R2=-20.6114, DA=0.5093, PnL=0.0220
LayerNorm: RMSE=0.0386, R2=-7.0566, DA=0.4888, PnL=0.0568

📊 Horizon t+3
NoNorm: RMSE=0.0173, R2=-0.6164, DA=0.5279, PnL=0.1741
BatchNorm: RMSE=0.0331, R2=-4.9132, DA=0.4796, PnL=-0.0325
LayerNorm: RMSE=0.0562, R2=-16.0657, DA=0.4963, PnL=0.1141

📊 Horizon t+5
NoNorm: RMSE=0.0234, R2=-1.9663, DA=0.4655, PnL=-0.1234
BatchNorm: RMSE=0.0695, R2=-25.1159, DA=0.5084, PnL=0.0105
LayerNorm: RMSE=0.1299, R2=-90.2446, DA=0.4860, PnL=-0.0237

🚀 Processing HDFCBANK


[*********************100%***********************]  1 of 1 completed


HDFCBANK usable rows: 2750

📊 Horizon t+1
NoNorm: RMSE=0.0262, R2=-3.7440, DA=0.4926, PnL=-0.0568
BatchNorm: RMSE=0.0369, R2=-8.4401, DA=0.5019, PnL=0.1576
LayerNorm: RMSE=0.1332, R2=-121.8074, DA=0.5390, PnL=0.2774

📊 Horizon t+3
NoNorm: RMSE=0.0283, R2=-4.5469, DA=0.5037, PnL=0.0897
BatchNorm: RMSE=0.0212, R2=-2.1031, DA=0.5019, PnL=0.0216
LayerNorm: RMSE=0.0796, R2=-42.8659, DA=0.5000, PnL=-0.0519

📊 Horizon t+5
NoNorm: RMSE=0.0233, R2=-2.7836, DA=0.4879, PnL=0.0511
BatchNorm: RMSE=0.0433, R2=-12.0341, DA=0.4916, PnL=-0.0830
LayerNorm: RMSE=0.2094, R2=-303.5374, DA=0.4935, PnL=0.0175

🚀 Processing INFOSYS


[*********************100%***********************]  1 of 1 completed


INFOSYS usable rows: 2750

📊 Horizon t+1
NoNorm: RMSE=0.0315, R2=-3.2516, DA=0.4796, PnL=-0.3027
BatchNorm: RMSE=0.0442, R2=-7.3519, DA=0.4963, PnL=0.2955
LayerNorm: RMSE=0.2505, R2=-267.5120, DA=0.4907, PnL=-0.2188

📊 Horizon t+3
NoNorm: RMSE=0.0255, R2=-1.7739, DA=0.4963, PnL=-0.0817
BatchNorm: RMSE=0.0203, R2=-0.7565, DA=0.5167, PnL=0.5882
LayerNorm: RMSE=0.1230, R2=-63.7512, DA=0.4851, PnL=-0.1509

📊 Horizon t+5
NoNorm: RMSE=0.0423, R2=-6.6850, DA=0.4972, PnL=0.2858
BatchNorm: RMSE=0.0386, R2=-5.3794, DA=0.4711, PnL=-0.7254
LayerNorm: RMSE=0.0310, R2=-3.1057, DA=0.4581, PnL=-0.9389
